# Practice 2 — Thực hành mô hình Neural Network đã huấn luyện trước

**Đề tài:** Phân loại ảnh CIFAR-10 bằng ResNet18 pretrained  
**Công cụ:** Python, PyTorch, Torchvision, Jupyter Notebook

## Mục tiêu
1. Tìm hiểu bộ dữ liệu CIFAR-10.
2. Dùng mô hình ResNet18 đã được huấn luyện trước trên ImageNet.
3. Thay lớp phân loại cuối để nhận diện 10 nhóm ảnh CIFAR-10.
4. Huấn luyện thử, đo Accuracy và xem biểu đồ Loss.

> Lưu ý: File ZIP bạn gửi có `cifar-10-python.npz` chỉ là tệp trỏ Git LFS (không chứa dữ liệu ảnh thật). Notebook này dùng `torchvision` tự tải CIFAR-10 chính thức khi chạy lần đầu. Cần kết nối Internet.

## 1. Import thư viện

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import time

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", device)

ModuleNotFoundError: No module named 'torch'

## 2. Chuẩn bị bộ dữ liệu CIFAR-10
CIFAR-10 có 60.000 ảnh màu thuộc 10 lớp: 50.000 ảnh train và 10.000 ảnh test. ResNet18 pretrained cần ảnh đầu vào 224×224, nên ta resize ảnh từ 32×32.

In [ ]:
classes = ['máy bay', 'ô tô', 'chim', 'mèo', 'hươu',
           'chó', 'ếch', 'ngựa', 'tàu thủy', 'xe tải']

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_all = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_all = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

# Lấy một phần nhỏ để chạy demo nhanh trên máy cá nhân.
# Có thể tăng số lượng sau khi notebook chạy ổn.
train_set, _ = torch.utils.data.random_split(
    train_all, [2000, len(train_all) - 2000],
    generator=torch.Generator().manual_seed(42)
)
test_set, _ = torch.utils.data.random_split(
    test_all, [500, len(test_all) - 500],
    generator=torch.Generator().manual_seed(42)
)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=32, shuffle=True, num_workers=0
)
test_loader = torch.utils.data.DataLoader(
    test_set, batch_size=32, shuffle=False, num_workers=0
)

print("Số ảnh dùng để huấn luyện:", len(train_set))
print("Số ảnh dùng để kiểm tra:", len(test_set))

## 3. Xem thử một số ảnh

In [ ]:
# Hiển thị ảnh trước khi chuẩn hóa để dễ quan sát
preview = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=False, transform=transforms.ToTensor()
)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image, label = preview[i]
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Nạp mô hình ResNet18 pretrained
`weights=ResNet18_Weights.DEFAULT` tải trọng số đã học từ ImageNet. Ta giữ phần trích xuất đặc trưng và thay lớp cuối để dự đoán 10 lớp CIFAR-10.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# Đóng băng các lớp trích xuất đặc trưng để demo huấn luyện nhanh.
for name, parameter in model.named_parameters():
    if not name.startswith("fc."):
        parameter.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Đã nạp ResNet18 pretrained và thay lớp phân loại cuối.")

## 5. Huấn luyện mô hình
Notebook dùng 2 epoch và 2.000 ảnh để minh họa quy trình. Đây là bản demo phục vụ thực hành/thuyết trình, không phải kết quả tối ưu cuối cùng.

In [ ]:
epochs = 2
loss_history = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    start = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Thời gian: {time.time()-start:.1f}s")

## 6. Đánh giá bằng Accuracy trên tập test

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        predictions = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy trên tập test: {accuracy:.2f}%")

## 7. Vẽ biểu đồ Loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
plt.title("ResNet18 - Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(range(1, len(loss_history) + 1))
plt.grid(True)
plt.show()

## 8. Thử dự đoán một ảnh

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images_device = images.to(device)

with torch.no_grad():
    outputs = model(images_device)
    predictions = outputs.argmax(dim=1).cpu()

# Ảnh đã chuẩn hóa nên khôi phục gần đúng màu ban đầu để hiển thị
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    img = images[i] * std + mean
    img = img.clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(f"Dự đoán: {classes[predictions[i]]}\nThật: {classes[labels[i]]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


KeyboardInterrupt

